# Dripito Rev-B — Validation Analysis

Reproducible pipeline that processes raw EXP-3 run CSVs and ground-truth
gravimetric measurements into the validation figures cited in
`docs/results.md`. Designed to run headless via `papermill` from
`docker compose up regenerate-figures` (or `regenerate-figures-sample`).

**Inputs**

- per-run device CSVs (in DATA_DIR or DATA_DIR/raw depending on SAMPLE_MODE)
- `geometry.json` from EXP-1
- `gravimetric_log.csv` from bench logging

**Outputs**

- `fig_bland_altman_combined.png`
- `fig_bland_altman_per_rate.png`
- `fig_mape_table.png`
- `fig_error_vs_flow.png`
- `fig_drop_volume_distribution.png`
- `results_summary.csv`, `mape_summary.csv`

In [ ]:
# Parameters (overridable by papermill -p)
DATA_DIR = "../../data/sample"
FIGURES_DIR = "../figures"
SAMPLE_MODE = True  # True -> CSVs in DATA_DIR; False -> CSVs in DATA_DIR/raw
TRAIN_PER_RATE = 4  # falls back to fit-on-all if any rate has fewer
BOOTSTRAP_ITER = 10000
RANDOM_SEED = 23

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make scripts importable whether running from notebooks/ or analysis/
ANALYSIS_ROOT = Path.cwd().resolve()
if (ANALYSIS_ROOT / "scripts").is_dir():
    sys.path.insert(0, str(ANALYSIS_ROOT))
else:
    sys.path.insert(0, str(ANALYSIS_ROOT.parent))

from scripts.load_run import load_run, summarise_run
from scripts.calibration_fit import (
    apply_correction, fit_correction_factor,
    gravimetric_flow_mlh, stratified_split,
)
from scripts.bland_altman import bland_altman_combined, bland_altman_per_rate
from scripts.bootstrap_mape import bootstrap_mape, mape

DATA_DIR_P = Path(DATA_DIR).resolve()
FIGURES_DIR_P = Path(FIGURES_DIR).resolve()
FIGURES_DIR_P.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    = {DATA_DIR_P}")
print(f"FIGURES_DIR = {FIGURES_DIR_P}")
print(f"SAMPLE_MODE = {SAMPLE_MODE}")

## 1. Load geometry and ground-truth log

In [ ]:
geometry = json.loads((DATA_DIR_P / "geometry.json").read_text())
beam_mm = geometry["beam_separation_mm"]["mean"]
print(f"Beam separation: {beam_mm} mm (sd={geometry['beam_separation_mm']['sd']} mm)")

grav = pd.read_csv(DATA_DIR_P / "gravimetric_log.csv")
grav["gravimetric_flow_mlh"] = grav.apply(
    lambda r: gravimetric_flow_mlh(r["gravimetric_mass_g"], r["run_duration_s"]),
    axis=1,
)
print(f"\n{len(grav)} runs in gravimetric_log.csv")
grav

## 2. Process each run CSV

In [ ]:
runs_dir = DATA_DIR_P if SAMPLE_MODE else (DATA_DIR_P / "raw")

run_records = []
all_drops = []  # per-drop rows across all runs for distribution plots
for _, g in grav.iterrows():
    csv_path = runs_dir / g["csv_filename"]
    if not csv_path.exists():
        print(f"  SKIP {g['run_id']}: {csv_path.name} not found")
        continue
    drops_df = load_run(csv_path, beam_mm)
    summary = summarise_run(drops_df, g["run_duration_s"])
    run_records.append({
        "run_id": g["run_id"],
        "flow_rate_target_mlh": g["flow_rate_target_mlh"],
        "duration_s": g["run_duration_s"],
        "drop_count_device": summary["drop_count"],
        "drop_count_log": g["drop_count_device"],
        "mean_drop_volume_uL": summary["mean_drop_volume_uL"],
        "mean_velocity_m_s": summary["mean_velocity_m_s"],
        "transit_us_cv": summary["transit_us_cv"],
        "device_flow_mlh": summary["device_flow_mlh"],
        "gravimetric_flow_mlh": g["gravimetric_flow_mlh"],
    })
    drops_df["run_id"] = g["run_id"]
    drops_df["flow_rate_target_mlh"] = g["flow_rate_target_mlh"]
    all_drops.append(drops_df)

runs_df = pd.DataFrame(run_records)
drops_df = pd.concat(all_drops, ignore_index=True) if all_drops else pd.DataFrame()
print(f"Processed {len(runs_df)} runs, {len(drops_df)} total drops")
runs_df

## 3. Fit scalar correction factor k

The dual-beam volume model has a systematic bias (chord length ≠ drop
diameter; beam-width effects). One scalar `k` absorbs this bias.

If N per flow rate >= TRAIN_PER_RATE we do a stratified train/test split
and report out-of-sample MAPE. Otherwise (sparse sample data) fit on
all runs and flag it explicitly.

In [ ]:
min_per_rate = runs_df.groupby("flow_rate_target_mlh").size().min()
use_split = min_per_rate >= TRAIN_PER_RATE
print(f"Min runs per rate: {min_per_rate}; TRAIN_PER_RATE: {TRAIN_PER_RATE}; using split: {use_split}")

if use_split:
    train_df, test_df = stratified_split(runs_df, train_per_rate=TRAIN_PER_RATE, seed=RANDOM_SEED)
    k = fit_correction_factor(train_df)
    print(f"\nFit k = {k:.4f} on {len(train_df)} TRAIN runs ({len(test_df)} TEST runs held out)")
    runs_df = apply_correction(runs_df, k)
    train_df = apply_correction(train_df, k)
    test_df = apply_correction(test_df, k)
else:
    k = fit_correction_factor(runs_df)
    print(f"\nFit k = {k:.4f} on all {len(runs_df)} runs (sparse data — no train/test split)")
    runs_df = apply_correction(runs_df, k)
    train_df, test_df = runs_df, runs_df

    runs_df["error_mlh"] = runs_df["device_flow_corrected_mlh"] - runs_df["gravimetric_flow_mlh"]
runs_df["error_mlh"] = runs_df["device_flow_corrected_mlh"] - runs_df["gravimetric_flow_mlh"]
runs_df["error_pct"] = runs_df["error_mlh"] / runs_df["gravimetric_flow_mlh"] * 100
runs_df[["run_id", "flow_rate_target_mlh", "device_flow_corrected_mlh", "gravimetric_flow_mlh", "error_pct"]]

## 4. Bland-Altman plots

In [ ]:
fig = bland_altman_combined(runs_df, "device_flow_corrected_mlh", "gravimetric_flow_mlh")
fig.savefig(FIGURES_DIR_P / "fig_bland_altman_combined.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
fig = bland_altman_per_rate(runs_df, "device_flow_corrected_mlh", "gravimetric_flow_mlh")
fig.savefig(FIGURES_DIR_P / "fig_bland_altman_per_rate.png", dpi=140, bbox_inches="tight")
plt.show()

## 5. MAPE with 95% bootstrap CI per flow rate

In [ ]:
mape_rows = []
for rate, sub in runs_df.groupby("flow_rate_target_mlh"):
    if len(sub) >= 1:
        result = bootstrap_mape(
            sub["device_flow_corrected_mlh"].to_numpy(),
            sub["gravimetric_flow_mlh"].to_numpy(),
            n_iter=BOOTSTRAP_ITER, seed=RANDOM_SEED,
        )
        mape_rows.append({"flow_rate_mlh": rate, "n": result["n"],
                          "mape_pct": result["mape"],
                          "ci_low_pct": result["ci_lower"],
                          "ci_high_pct": result["ci_upper"]})
combined_result = bootstrap_mape(
    runs_df["device_flow_corrected_mlh"].to_numpy(),
    runs_df["gravimetric_flow_mlh"].to_numpy(),
    n_iter=BOOTSTRAP_ITER, seed=RANDOM_SEED,
)
mape_rows.append({"flow_rate_mlh": "ALL", "n": combined_result["n"],
                  "mape_pct": combined_result["mape"],
                  "ci_low_pct": combined_result["ci_lower"],
                  "ci_high_pct": combined_result["ci_upper"]})
mape_df = pd.DataFrame(mape_rows)
mape_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.5 + 0.4 * len(mape_df)), dpi=140)
ax.axis("off")
cell_text = [[str(r["flow_rate_mlh"]), str(r["n"]),
              f"{r['mape_pct']:.2f}",
              f"[{r['ci_low_pct']:.2f}, {r['ci_high_pct']:.2f}]"]
             for _, r in mape_df.iterrows()]
table = ax.table(cellText=cell_text,
                 colLabels=["Flow rate (mL/hr)", "N", "MAPE (%)", "95% CI"],
                 loc="center", cellLoc="center", colLoc="center")
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1, 1.5)
ax.set_title(f"MAPE with 95% bootstrap CI (n_iter={BOOTSTRAP_ITER})", pad=12)
fig.savefig(FIGURES_DIR_P / "fig_mape_table.png", dpi=140, bbox_inches="tight")
plt.show()

## 6. Error vs flow rate

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=140)
ax.scatter(runs_df["flow_rate_target_mlh"], runs_df["error_pct"],
           s=55, alpha=0.8, color="steelblue", edgecolor="black", linewidth=0.5)
if len(runs_df) >= 3:
    z = np.polyfit(runs_df["flow_rate_target_mlh"], runs_df["error_pct"], 1)
    xs = np.linspace(runs_df["flow_rate_target_mlh"].min(), runs_df["flow_rate_target_mlh"].max(), 50)
    ax.plot(xs, np.polyval(z, xs), color="firebrick", lw=1.2,
            label=f"trend: {z[0]:+.2f}% per mL/hr")
ax.axhline(0, color="lightgrey", lw=0.7)
ax.set_xlabel("Target flow rate (mL/hr)")
ax.set_ylabel("Error (%)  =  100 × (device − truth) / truth")
ax.set_title("Per-run error vs flow rate")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR_P / "fig_error_vs_flow.png", dpi=140, bbox_inches="tight")
plt.show()

## 7. Drop volume distribution per flow rate

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=140)
rates = sorted(drops_df["flow_rate_target_mlh"].unique())
data = [drops_df[drops_df["flow_rate_target_mlh"] == r]["drop_volume_uL"].to_numpy() for r in rates]
bp = ax.boxplot(data, tick_labels=[f"{r}" for r in rates],
                patch_artist=True, showmeans=True, medianprops={"color": "black"})
for patch, color in zip(bp["boxes"], plt.cm.viridis(np.linspace(0.3, 0.85, len(rates)))):
    patch.set_facecolor(color); patch.set_alpha(0.65)
ax.set_xlabel("Target flow rate (mL/hr)")
ax.set_ylabel("Computed drop volume (µL, sphere model, pre-correction)")
ax.set_title("Drop volume distribution per flow rate")
ax.grid(True, alpha=0.25, axis="y")
fig.tight_layout()
fig.savefig(FIGURES_DIR_P / "fig_drop_volume_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

## 8. Persist tabular summary

In [ ]:
summary_cols = ["run_id", "flow_rate_target_mlh", "drop_count_device", "duration_s",
                "mean_drop_volume_uL", "device_flow_mlh", "device_flow_corrected_mlh",
                "gravimetric_flow_mlh", "error_mlh", "error_pct"]
runs_df[summary_cols].to_csv(FIGURES_DIR_P / "results_summary.csv", index=False)
mape_df.to_csv(FIGURES_DIR_P / "mape_summary.csv", index=False)
print(f"Wrote {FIGURES_DIR_P / 'results_summary.csv'}")
print(f"Wrote {FIGURES_DIR_P / 'mape_summary.csv'}")
print(f"\nCorrection factor k = {k:.4f}")
print(f"Pipeline complete.")

---

## Provenance

This notebook is the **source of truth** for every validation number
in `docs/results.md` and the top-level `README.md`. Regenerate via:

```bash
cd analysis
docker compose up regenerate-figures-sample   # sample data
docker compose up regenerate-figures          # real campaign data
```

Both invocations execute this notebook headlessly via `papermill`
with pinned dependencies. RANDOM_SEED is fixed so re-runs are
byte-identical.